# Support Vector Machines

**Junior Design**

---

## Where We Are

So far we've covered:
- Linear & logistic regression
- Decision trees & random forests
- Cross-validation, gradient descent, bootstrapping/bagging/boosting
- Feature engineering, scaling, encoding, selection, PCA, pipelines

We still want to add (at least) two major families of models to your toolkit:

**Support Vector Machines (SVMs)** — a powerful approach to classification (and regression) that finds the optimal boundary between classes by maximizing the margin. SVMs were the dominant ML technique before deep learning took over, and they remain excellent for many problems, especially with small-to-medium datasets.

**Neural Networks** — the foundation of modern deep learning. We'll start from a single neuron (which is just logistic regression in disguise), build up to multilayer networks, and see how backpropagation connects to the gradient descent you already know.

Both of these are supported by **pyMAISE**, so this directly prepares you for that work.

### Today we'll focus on SVMs

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score
)
from sklearn.datasets import (
    make_moons, make_circles, make_classification,
    load_breast_cancer, load_digits
)
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("All imports successful!")

---

# PART I: SUPPORT VECTOR MACHINES

---

## Part 1: SVM Intuition — Margins & Support Vectors

### The Core Idea

Imagine you have two classes of points in 2D and you want to draw a line separating them. There are infinitely many lines that could do this. Which one is "best"?

**SVM's answer:** The best boundary is the one that maximizes the **margin** — the distance between the boundary and the nearest data points from each class.

The points that are closest to the boundary (and thus *define* it) are called **support vectors**. If you removed any other point from the training set, the boundary wouldn't change. Only the support vectors matter.

This is fundamentally different from logistic regression, which is influenced by *all* data points.

In [ ]:
# ============================================================
# Helper function: plot 2D decision boundaries
# We'll use this throughout the notebook
# ============================================================
def plot_decision_boundary(model, X, y, ax=None, title='', h=0.02, show_sv=False):
    """Plot the decision boundary of a 2D classifier."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict across the grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#0000FF'])
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, edgecolors='black', s=40, alpha=0.8)
    
    # Highlight support vectors if SVM
    if show_sv and hasattr(model, 'support_vectors_'):
        sv = model.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=200, facecolors='none', 
                   edgecolors='green', linewidths=2, label=f'Support Vectors (n={len(sv)})')
        ax.legend(fontsize=10)
    elif show_sv and hasattr(model, 'named_steps'):
        # For pipeline with SVM
        svm_step = model.named_steps.get('svm') or model.named_steps.get('model')
        if hasattr(svm_step, 'support_vectors_'):
            # Need to inverse-transform if scaled
            scaler = model.named_steps.get('scaler')
            sv = svm_step.support_vectors_
            if scaler:
                sv = scaler.inverse_transform(sv)
            ax.scatter(sv[:, 0], sv[:, 1], s=200, facecolors='none',
                       edgecolors='green', linewidths=2, label=f'Support Vectors (n={len(sv)})')
            ax.legend(fontsize=10)
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    return ax

print("Helper function ready.")

In [ ]:
# ============================================================
# Visualize the SVM concept: maximum margin classifier
# ============================================================
np.random.seed(42)

# Generate simple linearly separable data
X_simple, y_simple = make_classification(
    n_samples=80, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=2.0, random_state=42
)

# Compare: Logistic Regression vs SVM
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_simple, y_simple)
plot_decision_boundary(lr, X_simple, y_simple, ax=axes[0], 
                       title='Logistic Regression\n(influenced by ALL points)')

# SVM
svm_linear = SVC(kernel='linear')
svm_linear.fit(X_simple, y_simple)
plot_decision_boundary(svm_linear, X_simple, y_simple, ax=axes[1], 
                       title='SVM (Linear Kernel)\n(only support vectors matter)',
                       show_sv=True)

plt.suptitle('Logistic Regression vs SVM: Different Decision Philosophies', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"SVM uses {len(svm_linear.support_vectors_)} support vectors out of {len(X_simple)} training points.")
print("The boundary is defined ONLY by these points — all others could be removed\n"
      "without changing the decision boundary at all.")

### The Math (Brief Version)

For a linear SVM, the decision boundary is a hyperplane:

$$\vec{w} \cdot \vec{x} + b = 0$$

The SVM finds $\vec{w}$ and $b$ that maximize the margin $\frac{2}{\|\vec{w}\|}$ subject to all points being on the correct side:

$$y_i(\vec{w} \cdot \vec{x}_i + b) \geq 1 \quad \forall i$$

This is a **constrained optimization problem** — maximizing margin while correctly classifying everything. It can be solved efficiently using quadratic programming.

---

## Part 2: The Kernel Trick — Nonlinear Boundaries

Real data is rarely linearly separable. The **kernel trick** is what makes SVMs truly powerful.

**The idea:** If data isn't separable in the original space, project it into a *higher-dimensional* space where it becomes separable, then find the linear boundary *there*.

**The trick:** You never actually compute the high-dimensional coordinates. Kernels compute the *dot product* in the higher-dimensional space directly from the original coordinates. This is computationally efficient and mathematically elegant.

Common kernels:
- **Linear:** $K(x_i, x_j) = x_i \cdot x_j$ (no transformation)
- **Polynomial:** $K(x_i, x_j) = (\gamma \cdot x_i \cdot x_j + r)^d$
- **RBF (Gaussian):** $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$ — most commonly used

In [ ]:
# ============================================================
# Why we need nonlinear boundaries
# ============================================================

# Generate two datasets that are NOT linearly separable
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, X, y, name in [(axes[0], X_moons, y_moons, 'Moons'), 
                        (axes[1], X_circles, y_circles, 'Circles')]:
    cmap = ListedColormap(['Red', 'Blue'])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, edgecolors='black', s=40)
    ax.set_title(f'{name} Dataset — NOT Linearly Separable', fontweight='bold')

plt.tight_layout()
plt.show()
print("No straight line can separate either of these datasets.")
print("We need curved decision boundaries — this is where kernels come in.")

In [ ]:
# ============================================================
# Compare kernels on the moons dataset
# ============================================================
kernels = [
    ('Linear', SVC(kernel='linear')),
    ('Polynomial (degree=3)', SVC(kernel='poly', degree=3)),
    ('RBF (Gaussian)', SVC(kernel='rbf', gamma='scale')),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, kernels):
    model.fit(X_moons, y_moons)
    acc = model.score(X_moons, y_moons)
    plot_decision_boundary(model, X_moons, y_moons, ax=ax, 
                           title=f'{name}\nAccuracy: {acc:.1%}', show_sv=True)

plt.suptitle('SVM Kernels on the Moons Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# The kernel trick visualized: what RBF does conceptually
# ============================================================
# With the circles dataset, the idea is clearest.
# In 2D, the classes form concentric rings — not separable by a line.
# But if we add a 3rd dimension (e.g., x1² + x2²), the inner circle
# lifts UP and becomes separable by a flat plane!

fig = plt.figure(figsize=(16, 6))

# Original 2D view
ax1 = fig.add_subplot(121)
cmap = ListedColormap(['Red', 'Blue'])
ax1.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap=cmap, 
            edgecolors='black', s=40)
ax1.set_title('Original 2D Space\n(not linearly separable)', fontweight='bold')
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')

# Projected into 3D
ax2 = fig.add_subplot(122, projection='3d')
z = X_circles[:, 0]**2 + X_circles[:, 1]**2  # radial distance squared
colors = ['Red' if yi == 0 else 'Blue' for yi in y_circles]
ax2.scatter(X_circles[:, 0], X_circles[:, 1], z, c=colors, 
            edgecolors='black', s=40, alpha=0.8)

# Draw the separating plane
xx_plane, yy_plane = np.meshgrid(np.linspace(-1.2, 1.2, 10), np.linspace(-1.2, 1.2, 10))
z_threshold = 0.3  # approximate separating height
ax2.plot_surface(xx_plane, yy_plane, np.full_like(xx_plane, z_threshold), 
                 alpha=0.15, color='green')

ax2.set_title('Projected to 3D: $z = x_1^2 + x_2^2$\n(NOW linearly separable!)', fontweight='bold')
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_zlabel('$x_1^2 + x_2^2$')
ax2.view_init(elev=20, azim=45)

plt.tight_layout()
plt.show()

print("The kernel trick does this projection IMPLICITLY — it never actually")
print("computes the 3D coordinates. It only computes dot products in that space,")
print("which is all the SVM optimization needs.")

---

## Part 3: SVM Hyperparameters — C and Gamma

Two hyperparameters control an RBF SVM's behavior:

### C (Regularization)
Controls the trade-off between a smooth decision boundary and correctly classifying training points.
- **Small C** → wider margin, more misclassifications allowed (high bias, low variance)
- **Large C** → narrow margin, fewer misclassifications (low bias, high variance = overfitting risk)

### Gamma (RBF kernel width)
Controls how far the influence of a single training point reaches.
- **Small gamma** → each point influences a large area (smoother boundary)
- **Large gamma** → each point only influences nearby area (more complex, wiggly boundary)

In [ ]:
# ============================================================
# Visualize the effect of C
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, C in zip(axes, [0.01, 1.0, 1000]):
    svm = SVC(kernel='rbf', C=C, gamma='scale')
    svm.fit(X_moons, y_moons)
    acc = svm.score(X_moons, y_moons)
    plot_decision_boundary(svm, X_moons, y_moons, ax=ax,
                           title=f'C = {C}\nAccuracy: {acc:.1%}, SVs: {len(svm.support_vectors_)}',
                           show_sv=True)

plt.suptitle('Effect of C (Regularization Parameter)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Small C → soft margin, allows errors, simpler boundary")
print("Large C → hard margin, few errors allowed, complex boundary (overfitting risk)")

In [ ]:
# ============================================================
# Visualize the effect of Gamma
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, gamma in zip(axes, [0.1, 1.0, 50]):
    svm = SVC(kernel='rbf', C=1.0, gamma=gamma)
    svm.fit(X_moons, y_moons)
    acc = svm.score(X_moons, y_moons)
    plot_decision_boundary(svm, X_moons, y_moons, ax=ax,
                           title=f'gamma = {gamma}\nAccuracy: {acc:.1%}',
                           show_sv=True)

plt.suptitle('Effect of Gamma (RBF Kernel Width)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Small gamma → smooth, broad influence → underfitting")
print("Large gamma → tight, local influence → overfitting (boundary wraps around each point)")
print("\n Look at gamma=50: the boundary memorizes the training data.")
print("It would perform terribly on new data.")

---

## Part 4: SVMs on Real Data — Breast Cancer Classification

Let's apply SVMs to a real-world medical dataset. The **Wisconsin Breast Cancer** dataset contains measurements from cell nuclei in breast mass biopsies. The task: classify tumors as **malignant** or **benign** based on 30 features.

This is exactly the kind of problem SVMs excel at: moderate number of samples (~570), moderate features (30), binary classification, and the cost of misclassification is high.

In [ ]:
# ============================================================
# Load and explore the breast cancer dataset
# ============================================================
cancer = load_breast_cancer()
X_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_cancer = cancer.target  # 0 = malignant, 1 = benign

print(f"Dataset shape: {X_cancer.shape}")
print(f"Classes: {dict(zip(cancer.target_names, np.bincount(y_cancer)))}")
print(f"\nFeature examples (first 10):")
for name in cancer.feature_names[:10]:
    print(f"  {name}")
print(f"  ... and {len(cancer.feature_names) - 10} more")

print(f"\nFeature value ranges (showing scale differences):")
for feat in ['mean radius', 'mean texture', 'mean area', 'mean smoothness']:
    vals = X_cancer[feat]
    print(f"  {feat:<25s}  {vals.min():.2f} – {vals.max():.2f}")

In [ ]:
# ============================================================
# WHY SCALING MATTERS for SVMs
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

# Without scaling
svm_unscaled = SVC(kernel='rbf')
svm_unscaled.fit(X_train, y_train)
acc_unscaled = svm_unscaled.score(X_test, y_test)

# With scaling (using a pipeline!)
svm_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])
svm_scaled.fit(X_train, y_train)
acc_scaled = svm_scaled.score(X_test, y_test)

print("SVM Performance on Breast Cancer (Test Set):")
print(f"  Without scaling: {acc_unscaled:.1%}")
print(f"  With scaling:    {acc_scaled:.1%}")
print(f"\nScaling improved accuracy by {(acc_scaled - acc_unscaled)*100:.1f} percentage points!")
print("   This is because features like 'mean area' (~100-2500) completely")
print("   dominate features like 'mean smoothness' (~0.05-0.16) in the RBF kernel.")

In [ ]:
# ============================================================
# Compare SVM kernels on real data (with proper scaling)
# ============================================================
kernels_to_test = {
    'Linear': SVC(kernel='linear'),
    'Polynomial (d=3)': SVC(kernel='poly', degree=3),
    'RBF': SVC(kernel='rbf'),
}

print("SVM Kernel Comparison (5-fold CV on training set):")
print("=" * 50)

results = {}
for name, svm in kernels_to_test.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('svm', svm)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = scores
    print(f"  {name:<20s}  Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# ============================================================
# Detailed evaluation of the best SVM
# ============================================================
best_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True))
])
best_svm.fit(X_train, y_train)
y_pred = best_svm.predict(X_test)

print("Classification Report (Test Set):")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=cancer.target_names, 
    cmap='Blues', ax=ax
)
ax.set_title('SVM (RBF) — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Visualize: project data to 2D with PCA, then show SVM boundary
# ============================================================
# Since the real data has 30 features, we can't visualize it directly.
# Let's use PCA to project to 2D, then train an SVM on that.

pca = PCA(n_components=2)
X_cancer_2d = pca.fit_transform(StandardScaler().fit_transform(X_cancer))

print(f"Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.1%}")

svm_2d = SVC(kernel='rbf', gamma='scale')
svm_2d.fit(X_cancer_2d, y_cancer)

fig, ax = plt.subplots(figsize=(10, 7))
plot_decision_boundary(svm_2d, X_cancer_2d, y_cancer, ax=ax,
                       title=f'SVM (RBF) Decision Boundary — Breast Cancer (PCA to 2D)',
                       show_sv=True)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.tight_layout()
plt.show()

print("Red = Malignant, Blue = Benign")
print("Green circles = Support Vectors (the points that define the boundary)")